# 卷积神经网络

$$
\text{图像 Tensor}
\rightarrow
Conv2d
\rightarrow
ReLU
\rightarrow
Pooling
\rightarrow
Flatten
\rightarrow
Linear
\rightarrow
10类
$$

## 过程拆解

1. 数据集准备（同上节）

In [1]:
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data
y = digits.target

# 数据集划分
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(X_train_tensor.shape, X_train_tensor.dtype)
print(y_train_tensor.shape, y_train_tensor.dtype)

torch.Size([1437, 64]) torch.float32
torch.Size([1437]) torch.int64


2. 更改tensor形状
   PyTorch的`Conv2d`要求4维$(N,\ C,\ H,\ W)$，即样本数、通道数、高、宽。

In [2]:
X_train_cnn = X_train_tensor.reshape(-1, 1, 8, 8)
X_test_cnn = X_test_tensor.reshape(-1, 1, 8, 8)

print(X_train_cnn.shape)

torch.Size([1437, 1, 8, 8])


3. 定义卷积层

In [3]:
import torch.nn as nn
conv = nn.Conv2d(
    in_channels=1,
    out_channels=4,
    kernel_size=3,
    stride=1,
    padding=0
)

参数：
- in_channels=1：输入是灰度图，1 个通道
- out_channels=4：使用 4 个卷积核，所以输出 4 张 feature map
- kernel_size=3：每个卷积核是 $3\times3$，也可以不是正方形，即`kernel_size=(3, 5)`
- stride=1：同上，可可以用tuple分别给高和宽
- padding=0：同上

`Conv2d` 创建时会按照 PyTorch 的默认初始化方式初始化`weight`和`bias`。

In [4]:
print(conv.weight.shape)
print(conv.bias.shape)
print(conv.weight[0])

torch.Size([4, 1, 3, 3])
torch.Size([4])
tensor([[[ 0.0647,  0.1731,  0.3042],
         [-0.2295,  0.1775,  0.2230],
         [ 0.0962, -0.0057, -0.0591]]], grad_fn=<SelectBackward0>)


4. 一次forward

In [5]:
feature_linear = conv(X_train_cnn)
feature = torch.relu(feature_linear)
print(feature.shape)

torch.Size([1437, 4, 6, 6])


5. $2\times2$最大池化

In [6]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)

pooled = pool(feature)
print(pooled.shape)

torch.Size([1437, 4, 3, 3])


6. 全连接层
   
   为了将 CNN 得到的 $C\times H\times W$ 特征接入分类全连接层，将每个样本展平成一维特征向量。

In [7]:
flat = pooled.reshape(pooled.shape[0], -1)
print(flat.shape)

torch.Size([1437, 36])


In [8]:
fc = nn.Linear(36, 10)
logits = fc(flat)
print(logits.shape)

torch.Size([1437, 10])


7. 计算loss并反向传播

In [9]:
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, y_train_tensor)
print(loss)

loss.backward()

tensor(7.6106, grad_fn=<NllLossBackward0>)


8. 创建优化器并更新

In [10]:
optimizer = torch.optim.SGD(
    list(conv.parameters()) + list(fc.parameters()),
    lr=0.01
)

optimizer.step()
optimizer.zero_grad()

## 完整训练

In [11]:
def train(epochs, lr, adam=False, no_pool=False):
    torch.manual_seed(42)
    if no_pool:
        model = nn.Sequential(
            nn.Conv2d(1, 8, 3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(288, 10)
        )
    else:
        model = nn.Sequential(
            nn.Conv2d(1, 8, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(72, 10)
        )
    loss_fn = nn.CrossEntropyLoss()
    if adam:
        optimizer = torch.optim.Adam(model.parameters(), lr / 10)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr)
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(X_train_cnn)
        loss = loss_fn(logits, y_train_tensor)
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(loss)
    return model

In [12]:
model = train(500, 0.01)
model2 = train(500, 0.01, adam=True)
model3 = train(500, 0.01, no_pool=True)

tensor(7.3895, grad_fn=<NllLossBackward0>)
tensor(0.3061, grad_fn=<NllLossBackward0>)
tensor(0.1769, grad_fn=<NllLossBackward0>)
tensor(0.1298, grad_fn=<NllLossBackward0>)
tensor(0.1037, grad_fn=<NllLossBackward0>)
tensor(7.3895, grad_fn=<NllLossBackward0>)
tensor(0.5969, grad_fn=<NllLossBackward0>)
tensor(0.2296, grad_fn=<NllLossBackward0>)
tensor(0.1444, grad_fn=<NllLossBackward0>)
tensor(0.1030, grad_fn=<NllLossBackward0>)
tensor(4.0301, grad_fn=<NllLossBackward0>)
tensor(0.1567, grad_fn=<NllLossBackward0>)
tensor(0.1000, grad_fn=<NllLossBackward0>)
tensor(0.0745, grad_fn=<NllLossBackward0>)
tensor(0.0589, grad_fn=<NllLossBackward0>)


## 训练结果分析

In [13]:
with torch.no_grad():
    pred_train = torch.argmax(model(X_train_cnn), dim=1)
    pred_train2 = torch.argmax(model2(X_train_cnn), dim=1)
    pred_train3 = torch.argmax(model3(X_train_cnn), dim=1)
    train_acc = (
        (pred_train == y_train_tensor).float().mean(), 
        (pred_train2 == y_train_tensor).float().mean(),
        (pred_train3 == y_train_tensor).float().mean()
    )

print(train_acc)

with torch.no_grad():
    pred_test = torch.argmax(model(X_test_cnn), dim=1)
    pred_test2 = torch.argmax(model2(X_test_cnn), dim=1)
    pred_test3 = torch.argmax(model3(X_test_cnn), dim=1)
    acc_test = (
        (pred_test == y_test_tensor).float().mean(), 
        (pred_test2 == y_test_tensor).float().mean(),
        (pred_test3 == y_test_tensor).float().mean()
    )

print(acc_test)

(tensor(0.9812), tensor(0.9854), tensor(0.9916))
(tensor(0.9639), tensor(0.9694), tensor(0.9667))


## 总结

1. `nn.Sequential()`中装模型的各阶段，`optimizer = torch.optim.SGD(model.parameters(), lr)`为优化器。过程：
   ```
    optimizer.zero_grad()
    logits = model(X_train_cnn)
    loss = loss_fn(logits, y_train_tensor)
    loss.backward()
    optimizer.step()
   ```
3. 去掉 pooling 会保留更多空间细节，同时使 Flatten 后维度和全连接层参数量增加，因此训练集拟合能力可能增强，但泛化性能不一定提高。即测试集正确率可能下降。